# Fase 2: Limpieza y Consolidación

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql import functions as F

In [2]:
# Create the spark session
spark = SparkSession.builder \
    .appName("Bronze") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 05:46:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 05:46:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
# Read the data from each dir

df_source_1 = spark.read \
    .option("multiline", "true") \
    .json("/app/data/bronze/data_source_1") \
    .withColumn("user_id", F.lit("source_1"))

df_source_2 = spark.read \
    .option("multiline", "true") \
    .json("/app/data/bronze/data_source_2") \
    .withColumn("user_id", F.lit("source_2"))

In [4]:
print("--- Numero de registros en cada df ---")
print(f"Origen 1: {df_source_1.count():,}")
print(f"Origen 2: {df_source_2.count():,}")

--- Numero de registros en cada df ---


Origen 1: 262,824
Origen 2: 53,866


In [5]:
# Consolidate in a single DataFrame
df_consolidated = df_source_1.unionByName(df_source_2)

In [6]:
print(f"Numero de registros totales: {df_consolidated.count():,}")

[Stage 8:======================================>                  (14 + 7) / 21]

Numero de registros totales: 316,690


In [7]:
# Output the data as parquet
parquet_output_path = "../data/bronze/consolidated_raw.parquet"

df_consolidated.write \
    .mode("overwrite") \
    .parquet(parquet_output_path)

In [8]:
df_raw = spark.read.parquet("/app/data/bronze/consolidated_raw.parquet")

In [9]:
df_raw.printSchema()

root
 |-- audiobook_chapter_title: string (nullable = true)
 |-- audiobook_chapter_uri: string (nullable = true)
 |-- audiobook_title: string (nullable = true)
 |-- audiobook_uri: string (nullable = true)
 |-- conn_country: string (nullable = true)
 |-- episode_name: string (nullable = true)
 |-- episode_show_name: string (nullable = true)
 |-- incognito_mode: boolean (nullable = true)
 |-- ip_addr: string (nullable = true)
 |-- master_metadata_album_album_name: string (nullable = true)
 |-- master_metadata_album_artist_name: string (nullable = true)
 |-- master_metadata_track_name: string (nullable = true)
 |-- ms_played: long (nullable = true)
 |-- offline: boolean (nullable = true)
 |-- offline_timestamp: long (nullable = true)
 |-- platform: string (nullable = true)
 |-- reason_end: string (nullable = true)
 |-- reason_start: string (nullable = true)
 |-- shuffle: boolean (nullable = true)
 |-- skipped: boolean (nullable = true)
 |-- spotify_episode_uri: string (nullable = true)
 |

In [10]:
# Number of total registers

total_ingested = df_raw.count()

print(f"--- Numero de registros totales: {total_ingested:,} ---")

--- Numero de registros totales: 316,690 ---


In [11]:
# Drop the "ip_addr" column for data protection issues
df_raw = df_raw.drop("ip_addr")

In [12]:
# Show for the first 5 registers
df_raw.show(3, vertical=True)

-RECORD 0-------------------------------------------------
 audiobook_chapter_title           | NULL                 
 audiobook_chapter_uri             | NULL                 
 audiobook_title                   | NULL                 
 audiobook_uri                     | NULL                 
 conn_country                      | MX                   
 episode_name                      | NULL                 
 episode_show_name                 | NULL                 
 incognito_mode                    | false                
 master_metadata_album_album_name  | More Than a Friend   
 master_metadata_album_artist_name | girli                
 master_metadata_track_name        | More Than a Friend   
 ms_played                         | 17384                
 offline                           | true                 
 offline_timestamp                 | 1674740124           
 platform                          | ios                  
 reason_end                        | endplay            

In [13]:
# Count the original number of nulls
print("--- Conteo de valores NULL nativos por columna ---")
df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_raw.columns
]).show(vertical=True)

--- Conteo de valores NULL nativos por columna ---


[Stage 17:=========================>                               (5 + 6) / 11]

-RECORD 0-----------------------------------
 audiobook_chapter_title           | 316690 
 audiobook_chapter_uri             | 316690 
 audiobook_title                   | 316690 
 audiobook_uri                     | 316690 
 conn_country                      | 0      
 episode_name                      | 316470 
 episode_show_name                 | 316470 
 incognito_mode                    | 0      
 master_metadata_album_album_name  | 220    
 master_metadata_album_artist_name | 220    
 master_metadata_track_name        | 220    
 ms_played                         | 0      
 offline                           | 0      
 offline_timestamp                 | 140602 
 platform                          | 0      
 reason_end                        | 0      
 reason_start                      | 0      
 shuffle                           | 0      
 skipped                           | 0      
 spotify_episode_uri               | 316470 
 spotify_track_uri                 | 220    
 ts       

In [14]:
# Cleaning nulls
garbage_text_values = ["nan", "null", "na", "", "none", " "]
string_columns = [field.name for field in df_raw.schema if isinstance(field.dataType, StringType)]

df_cleaned = df_raw
for col_name in string_columns:
    df_cleaned = df_cleaned.withColumn(
        col_name,
        F.when(
            F.lower(F.trim(F.col(col_name))).isin(garbage_text_values) | F.col(col_name).isNull(),
            F.lit(None).cast(StringType())
        ).otherwise(F.col(col_name))
    )

df_cleaned = df_cleaned.dropna(subset=["ts", "ms_played"])

df_cleaned = df_cleaned.filter((F.col("ms_played").isNotNull()) & (F.col("ms_played") > 0))

null_fill_blueprint = {
    "master_metadata_track_name": "Unknown Track",
    "master_metadata_album_artist_name": "Unknown Artist",
    "master_metadata_album_album_name": "Unknown Album",
    "platform": "Unknown Platform",
    "conn_country": "Unknown Country",
    "episode_name": "Not A Podcast",
    "episode_show_name": "Not A Podcast",
    "spotify_episode_uri": "Not A Podcast",
    "reason_start": "unknown",
    "reason_end": "unknown",
    "offline_timestamp": 0
}

print(f"--- Numero de valores nulos dentro del dataset ---")
df_cleaned.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df_cleaned.columns
]).show(vertical=True)

df_cleaned = df_cleaned.fillna(null_fill_blueprint)

--- Numero de valores nulos dentro del dataset ---


[Stage 20:==========>                                              (2 + 9) / 11]

-RECORD 0-----------------------------------
 audiobook_chapter_title           | 309270 
 audiobook_chapter_uri             | 309270 
 audiobook_title                   | 309270 
 audiobook_uri                     | 309270 
 conn_country                      | 0      
 episode_name                      | 309054 
 episode_show_name                 | 309054 
 incognito_mode                    | 0      
 master_metadata_album_album_name  | 216    
 master_metadata_album_artist_name | 216    
 master_metadata_track_name        | 216    
 ms_played                         | 0      
 offline                           | 0      
 offline_timestamp                 | 135618 
 platform                          | 0      
 reason_end                        | 0      
 reason_start                      | 0      
 shuffle                           | 0      
 skipped                           | 0      
 spotify_episode_uri               | 309054 
 spotify_track_uri                 | 216    
 ts       

In [15]:
# Remove duplicated values
total_duplicates = total_ingested - df_cleaned.dropDuplicates(["ts", "user_id"]).count()
print(f"Total de registros duplicados detectados: {total_duplicates:,}")

df_cleaned = df_cleaned.dropDuplicates(["ts", "user_id"])

[Stage 23:====================>                                    (4 + 7) / 11]

Total de registros duplicados detectados: 52,158


In [16]:
# Add dates, year, month, day of the week, hour, and minutes columns to the df
df_cleaned = df_cleaned \
    .withColumn("date_timestamp", F.to_timestamp("ts")) \
    .withColumn("readable_date", F.to_date("date_timestamp")) \
    .withColumn("year", F.year("date_timestamp")) \
    .withColumn("month", F.month("date_timestamp")) \
    .withColumn("day_of_week", F.dayofweek("date_timestamp")) \
    .withColumn("hour", F.hour("date_timestamp")) \
    .withColumn("min_played", F.round(F.col("ms_played") / 60000, 2))

In [17]:
# Normalize song name, artist, and album name
text_targets = ["master_metadata_track_name", "master_metadata_album_artist_name", "master_metadata_album_album_name"]
for text_col in text_targets:
    df_cleaned = df_cleaned.withColumn(
        text_col,
        F.trim(F.regexp_replace(F.col(text_col), r"[\r\n\t]", " "))
    )

In [18]:
# Save cleaned DataFrame to parquet
df_cleaned.write.mode("overwrite").parquet("/app/data/silver/spotify_clean.parquet")

# Export to csv
df_cleaned.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .csv("/app/data/silver/spotify_clean_delivery")

In [19]:
print(f"Numero de registros despues de la limpieza: {df_cleaned.count():,}")

Numero de registros despues de la limpieza: 264,532


In [20]:
# Display of the data after being cleaned
df_cleaned.show(3, vertical=True)

[Stage 41:==========>                                              (2 + 9) / 11]

-RECORD 0-------------------------------------------------
 audiobook_chapter_title           | NULL                 
 audiobook_chapter_uri             | NULL                 
 audiobook_title                   | NULL                 
 audiobook_uri                     | NULL                 
 conn_country                      | MX                   
 episode_name                      | Not A Podcast        
 episode_show_name                 | Not A Podcast        
 incognito_mode                    | false                
 master_metadata_album_album_name  | What A Feeling       
 master_metadata_album_artist_name | Alex Gaudino         
 master_metadata_track_name        | What A Feeling - ... 
 ms_played                         | 57398                
 offline                           | false                
 offline_timestamp                 | 0                    
 platform                          | iOS 8.4 (iPad4,4)    
 reason_end                        | endplay            